In [5]:
# Load dataset
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

tweets = pd.read_csv('/content/drive/MyDrive/cleaned_sampled_tweets.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
print(tweets.head())

   target         ids                       date      flag          user  \
0       4  2048720362  2009-06-05 15:39:50-07:00  NO_QUERY      Judy_bcn   
1       0  2249549259  2009-06-19 22:32:38-07:00  NO_QUERY  msbunndotcom   
2       4  1957641384  2009-05-29 01:08:12-07:00  NO_QUERY    sarah_jean   
3       4  1961116118  2009-05-29 09:00:36-07:00  NO_QUERY     FreeBeing   
4       4  1963691694  2009-05-29 13:06:13-07:00  NO_QUERY     NinaMcFLY   

                                                text weekday  
0  @user ou sorry! my english is bad too :$ hehe ...  Friday  
1  just witnessed how easily my husband got my 2y...  Friday  
2               @user no, trying to meet baby spice!  Friday  
3  #FF one of my new faves cos y'all know how I a...  Friday  
4  it's cruel that I'm not living in the UK. wann...  Friday  


In [8]:
# Run sentiment analysis
from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model=(
        "cardiffnlp/"
        "twitter-roberta-base-sentiment-latest"
    ),
    top_k=None
)

results = sentiment_model(
    tweets["text"].tolist(),
    truncation=True,
    batch_size=50
)

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [12]:
# Put the results together
def scores_to_dict(scores):
    return {
        item["label"].lower(): item["score"]
        for item in scores
    }

score_dicts = [
    scores_to_dict(scores)
    for scores in results
]


tweets["sentiment_negative"] = [
    scores.get("negative", 0)
    for scores in score_dicts
]

tweets["sentiment_neutral"] = [
    scores.get("neutral", 0)
    for scores in score_dicts
]

tweets["sentiment_positive"] = [
    scores.get("positive", 0)
    for scores in score_dicts
]

def predicted_label(scores):
    return max(
        scores,
        key=scores.get
    ).capitalize()


tweets["sentiment"] = [
    predicted_label(scores)
    for scores in score_dicts
]

tweets["sentiment_score"] = (
    tweets["sentiment_positive"] - tweets["sentiment_negative"]
)

print(tweets.head())
print(tweets.tail())

   target         ids                       date      flag          user  \
0       4  2048720362  2009-06-05 15:39:50-07:00  NO_QUERY      Judy_bcn   
1       0  2249549259  2009-06-19 22:32:38-07:00  NO_QUERY  msbunndotcom   
2       4  1957641384  2009-05-29 01:08:12-07:00  NO_QUERY    sarah_jean   
3       4  1961116118  2009-05-29 09:00:36-07:00  NO_QUERY     FreeBeing   
4       4  1963691694  2009-05-29 13:06:13-07:00  NO_QUERY     NinaMcFLY   

                                                text weekday  \
0  @user ou sorry! my english is bad too :$ hehe ...  Friday   
1  just witnessed how easily my husband got my 2y...  Friday   
2               @user no, trying to meet baby spice!  Friday   
3  #FF one of my new faves cos y'all know how I a...  Friday   
4  it's cruel that I'm not living in the UK. wann...  Friday   

   sentiment_negative  sentiment_neutral  sentiment_positive sentiment  \
0            0.632282           0.328931            0.038786  Negative   
1         

In [16]:
# Get the sentiment by day of week data
sentiment_counts = (
    tweets.groupby(["weekday", "sentiment"])
    .size()
    .unstack(fill_value=0)
)

print(sentiment_counts)
print(sentiment_counts.shape)

sentiment  Negative  Neutral  Positive
weekday                               
Friday           91       61        98
Monday           79       63       108
Saturday         81       73        96
Sunday           78       68       104
Thursday        115       60        75
Tuesday          97       62        91
Wednesday       124       68        58
(7, 3)


In [18]:
# Save the results
tweets.to_csv("/content/drive/MyDrive/analyzed_tweets.csv", index=False)
sentiment_counts.to_csv("/content/drive/MyDrive/sentiment_by_weekdays.csv", index=False)
sentiment_counts.to_json("/content/drive/MyDrive/sentiment_by_weekdays.json")